In [ ]:
"""
Deploy the REST API to API Gateway + Lambda.
Uses OpenAPI import so all routes are visible in the API Gateway console.
"""

import boto3
import json
import time
import zipfile
import io
import copy

REGION = "us-east-1"
ACCOUNT_ID = "869795389844"
ROLE_ARN = "arn:aws:iam::869795389844:role/AgentCoreDemoRole"

LAMBDA_NAME = "RestAPI"
API_NAME = "RestAPI"
STAGE_NAME = "prod"

# --- Step 1: Create Lambda function ---

print("Creating Lambda function...")

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    with open("rest_api/lambda_function.py", "r") as f:
        zf.writestr("lambda_function.py", f.read())
zip_buffer.seek(0)

lam = boto3.client("lambda", region_name=REGION)

try:
    lam.delete_function(FunctionName=LAMBDA_NAME)
    print("  Deleted existing function")
except lam.exceptions.ResourceNotFoundException:
    pass

func = lam.create_function(
    FunctionName=LAMBDA_NAME,
    Runtime="python3.12",
    Role=ROLE_ARN,
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": zip_buffer.read()},
    Timeout=30,
    MemorySize=128,
)
lambda_arn = func["FunctionArn"]
print(f"  Lambda ARN: {lambda_arn}")

apigw = boto3.client("apigateway", region_name=REGION)

# --- Step 2: Build OpenAPI spec with API Gateway extensions ---

# Clean up any existing API Gateways with the same name
print("\nCleaning up existing API Gateways...")
existing_apis = apigw.get_rest_apis(limit=500).get("items", [])
for api in existing_apis:
    if api["name"] in (API_NAME, "REST API"):
        aid = api["id"]
        print(f"  Deleting API: {aid} ({api['name']})")

        # Remove usage plans linked to this API
        try:
            plans = apigw.get_usage_plans(limit=500).get("items", [])
            for plan in plans:
                for stage in plan.get("apiStages", []):
                    if stage.get("apiId") == aid:
                        # Remove API keys from the usage plan
                        keys = apigw.get_usage_plan_keys(usagePlanId=plan["id"], limit=500).get("items", [])
                        for k in keys:
                            apigw.delete_usage_plan_key(usagePlanId=plan["id"], keyId=k["id"])
                            apigw.delete_api_key(apiKey=k["id"])
                            print(f"    Deleted API key: {k['id']}")
                        # Remove the API stage from the usage plan
                        apigw.update_usage_plan(
                            usagePlanId=plan["id"],
                            patchOperations=[
                                {"op": "remove", "path": f"/apiStages", "value": f"{aid}:{STAGE_NAME}"}
                            ],
                        )
                        apigw.delete_usage_plan(usagePlanId=plan["id"])
                        print(f"    Deleted usage plan: {plan['id']}")
                        break
        except Exception as e:
            print(f"    Warning cleaning up usage plans: {e}")

        # Delete stage and API
        try:
            apigw.delete_stage(restApiId=aid, stageName=STAGE_NAME)
        except Exception:
            pass
        try:
            apigw.delete_rest_api(restApiId=aid)
            print(f"  Deleted API: {aid}")
        except Exception as e:
            print(f"  Failed to delete {aid}: {e}")
        time.sleep(1)

print("\nPreparing OpenAPI spec with API Gateway integrations...")

with open("rest_api/openapi.json", "r") as f:
    spec = json.load(f)

# The Lambda integration extension for API Gateway
lambda_integration = {
    "x-amazon-apigateway-integration": {
        "type": "aws_proxy",
        "httpMethod": "POST",
        "uri": f"arn:aws:apigateway:{REGION}:lambda:path/2015-03-31/functions/{lambda_arn}/invocations",
        "passthroughBehavior": "when_no_match",
    }
}

# Add the integration to every operation in the spec
apigw_spec = copy.deepcopy(spec)
for path, methods in apigw_spec["paths"].items():
    for method, operation in methods.items():
        operation.update(lambda_integration)

# Set the server URL (will be replaced after deployment)
apigw_spec["servers"] = [{"url": f"https://placeholder.execute-api.{REGION}.amazonaws.com/{STAGE_NAME}"}]

# --- Step 3: Import into API Gateway ---

print("\nCreating API Gateway from OpenAPI spec...")

api = apigw.import_rest_api(
    failOnWarnings=False,
    body=json.dumps(apigw_spec),
    parameters={
        "endpointConfigurationTypes": "REGIONAL",
    },
)
api_id = api["id"]
print(f"  API ID: {api_id}")
print(f"  API Name: {api['name']}")

# Grant API Gateway permission to invoke Lambda
try:
    lam.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="apigateway-invoke",
        Action="lambda:InvokeFunction",
        Principal="apigateway.amazonaws.com",
        SourceArn=f"arn:aws:execute-api:{REGION}:{ACCOUNT_ID}:{api_id}/*",
    )
except lam.exceptions.ResourceConflictException:
    pass

# Deploy to stage
apigw.create_deployment(restApiId=api_id, stageName=STAGE_NAME)
api_url = f"https://{api_id}.execute-api.{REGION}.amazonaws.com/{STAGE_NAME}"
print(f"  API URL: {api_url}")

# --- Step 3b: Set up API key requirement ---

print("\nSetting up API key...")

# Create API key
key = apigw.create_api_key(name="RestAPIKey", enabled=True)
api_key_value = key["value"]
api_key_id = key["id"]
print(f"  API Key ID: {api_key_id}")
print(f"  API Key Value: {api_key_value}")

# Create usage plan linked to the stage
plan = apigw.create_usage_plan(
    name="RestAPIPlan",
    apiStages=[{"apiId": api_id, "stage": STAGE_NAME}],
)
apigw.create_usage_plan_key(
    usagePlanId=plan["id"],
    keyId=api_key_id,
    keyType="API_KEY",
)

# Require API key on all methods
resources = apigw.get_resources(restApiId=api_id, limit=500)
for resource in resources["items"]:
    for method in resource.get("resourceMethods", {}).keys():
        apigw.update_method(
            restApiId=api_id,
            resourceId=resource["id"],
            httpMethod=method,
            patchOperations=[
                {"op": "replace", "path": "/apiKeyRequired", "value": "true"}
            ],
        )
        time.sleep(0.5)

# Redeploy with API key requirement
time.sleep(5)
apigw.create_deployment(restApiId=api_id, stageName=STAGE_NAME)
print("  API key required on all methods")

print("Waiting 5s for deployment to propagate...")
time.sleep(5)

# --- Step 4: Quick test ---

print("\nTesting API via Lambda invoke...")

test_endpoints = [
    {"httpMethod": "GET", "path": "/sports"},
    {"httpMethod": "GET", "path": "/sports/football/events"},
    {"httpMethod": "GET", "path": "/events/evt-001/odds"},
    {"httpMethod": "GET", "path": "/events/live"},
    {"httpMethod": "GET", "path": "/account/balance", "queryStringParameters": {"accountId": "default"}},
    {"httpMethod": "GET", "path": "/promotions"},
]

for ep in test_endpoints:
    event = {"httpMethod": ep["httpMethod"], "path": ep["path"], "pathParameters": {}, "queryStringParameters": ep.get("queryStringParameters", {}), "body": None}
    resp = lam.invoke(FunctionName=LAMBDA_NAME, Payload=json.dumps(event))
    result = json.loads(resp["Payload"].read())
    body = json.loads(result["body"])
    print(f"  {ep['httpMethod']} {ep['path']} -> {result['statusCode']} OK")

# --- Step 5: Update OpenAPI spec with actual URL and upload to S3 ---

print("\nUploading OpenAPI spec to S3...")

spec["servers"] = [{"url": api_url}]

with open("rest_api/openapi_deployed.json", "w") as f:
    json.dump(spec, f, indent=2)

s3 = boto3.client("s3", region_name=REGION)
bucket_name = f"agentcore-rest-api-{ACCOUNT_ID}"

try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"  Created S3 bucket: {bucket_name}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"  S3 bucket already exists: {bucket_name}")

s3.put_object(
    Bucket=bucket_name,
    Key="openapi.json",
    Body=json.dumps(spec),
    ContentType="application/json",
)
s3_uri = f"s3://{bucket_name}/openapi.json"
print(f"  Uploaded spec to: {s3_uri}")

# --- Step 6: Update AgentCore Gateway API Key ---

print("\nUpdating AgentCore Gateway API key credential...")

CREDENTIAL_PROVIDER_NAME = "rest-api-key"

agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

try:
    response = agentcore_control.update_api_key_credential_provider(
        name=CREDENTIAL_PROVIDER_NAME,
        apiKey=api_key_value,
    )
    print(f"  Updated credential provider: {CREDENTIAL_PROVIDER_NAME}")
    print(f"  Provider ARN: {response['apiKeyCredentialProviderItem']['apiKeyCredentialProviderArn']}")
except agentcore_control.exceptions.ResourceNotFoundException:
    # Create it if it doesn't exist yet
    print(f"  Credential provider '{CREDENTIAL_PROVIDER_NAME}' not found, creating...")
    response = agentcore_control.create_api_key_credential_provider(
        name=CREDENTIAL_PROVIDER_NAME,
        apiKey=api_key_value,
    )
    print(f"  Created credential provider: {CREDENTIAL_PROVIDER_NAME}")
    print(f"  Provider ARN: {response['apiKeyCredentialProviderItem']['apiKeyCredentialProviderArn']}")
except Exception as e:
    print(f"  Warning: Could not update AgentCore credential: {e}")
    print(f"  You may need to update it manually in the AgentCore console.")

# --- Done ---

print(f"\n{'='*60}")
print(f"REST API URL:  {api_url}")
print(f"API Key Value: {api_key_value}")
print(f"S3 OpenAPI:    {s3_uri}")
print(f"AgentCore credential provider: {CREDENTIAL_PROVIDER_NAME}")
print(f"API Gateway console: https://{REGION}.console.aws.amazon.com/apigateway/main/apis/{api_id}/resources?region={REGION}")
print(f"{'='*60}")